In [ ]:
########################################
#getting system arguments
import sys
def GetArg_dataName(default="Variables"):
    """
    #non-perturbations
    #Run One: python Tracked_Profiles.py Variables
    #Run Two: python Tracked_Profiles.py Entrainment
    #Run Three: python Tracked_Profiles.py PROCESSED_Entrainment (also can add _DivideMassFlux)
    #Run Four: python Tracked_Profiles.py W_Budgets
    #Run Five: python Tracked_Profiles.py QV_Budgets
    #Run Six: python Tracked_Profiles.py TH_Budgets

    #perturbations
    #Run One: python Tracked_Profiles.py Variables_Perturbations
    """
    # If run inside Jupyter, sys.argv will include ipykernel arguments
    if any("ipykernel_launcher" in arg for arg in sys.argv):
        print(f"Using default dataName: {default}")
        return default

    # If a user-specified argument exists, use it
    if len(sys.argv) > 1:
        out=sys.argv[1]
        print(f"Using argument dataName: {out}")
        return out

    return default

dataName = GetArg_dataName()

In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm
import copy
import warnings

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=3)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="Tracked_Profiles", dataName="Tracked_Ascent_Trajectories",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS

In [ ]:
#IMPORT FUNCTIONS

import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def Get_Lagrangian_Binary_Array_Data(ModelData, pIndices, varNames=None): 
    """
    Version for slicing a range of specific parcels
    """
    directory_Lagrangian_Binary_Array = f"/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/LagrangianArrays/Simulation_{ModelData.simulationNumber}_
    {ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz/Lagrangian_Binary_Array/"
    def limitParcels(ds): 
        ds = ds.isel(phony_dim_0=pIndices)
        if varNames is not None:
                ds = ds[varNames]
        return ds
    Lagrangian_Binary_Array_Data = OpenMultipleSingleTimes_LagrangianArray_JobArray(directory=directory_Lagrangian_Binary_Array,
                                                                                    ModelData=ModelData, start_job=None,end_job=None, 
                                                                                    limitParcels=limitParcels)
    return Lagrangian_Binary_Array_Data

# def Get_Lagrangian_Binary_Array_Data(ModelData, start_job,end_job):
#     """
#     Version for slicing a range of jobs
#     """
#     directory_Lagrangian_Binary_Array = f"/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/Code/OUTPUT/Variable_Calculation/LagrangianArrays/Simulation_{ModelData.simulationNumber}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz/Lagrangian_Binary_Array/"
#     Lagrangian_Binary_Array_Data,files = OpenMultipleSingleTimes_LagrangianArray_JobArray(directory_Lagrangian_Binary_Array, ModelData, start_job,end_job)
#     return Lagrangian_Binary_Array_Data

def GetSpatialData(Lagrangian_Binary_Array_Data):
    ds = Lagrangian_Binary_Array_Data.compute()

    # Now extract NumPy arrays
    W = ds['W'].data
    QCQI = ds['QCQI'].data
    parcel_z = ds['z'].data

    return W, QCQI, parcel_z
  
def OpenMultipleSingleTimes_LagrangianArray_JobArray_SpecifyFiles(files, ModelData, start_job,end_job, 
                                                     limitParcels=None,varNames=None,
                                                     pattern="Lagrangian_Binary_Array_*.h5"):
    """
    Load a sequence of Lagrangian .h5 files (each a single timestep)
    into one xarray.Dataset with dimensions (time, p),
    enforcing time order from ModelData.timeStrings.
    """

    # --- Open and concatenate along time
    if limitParcels is None:
        pIndices=slice(start_job,end_job)
        def limitParcelsFunction(ds): 
            ds = ds.isel(phony_dim_0=pIndices)
            if varNames is not None:
                ds = ds[varNames]
            return ds
    else:
        limitParcelsFunction = limitParcels
    chunk_spec = {'phony_dim_0': -1}  #new
    ds = xr.open_mfdataset(
        files,
        engine="h5netcdf",
        phony_dims="sort",
        combine="nested",
        concat_dim="time",
        preprocess=limitParcelsFunction,
    
        # 1. Stop xarray from generating tasks to check phony_dim_0 alignment across 661 files
        compat="override", #new
        coords="minimal", #new
        join="override", #new
    
        # 2. Chunking
        chunks=chunk_spec, #new
    
        # 3. Parallelize
        parallel=True #new
    )

    # --- Rename the phony dimension to 'p'
    if "phony_dim_0" in ds.dims:
        ds = ds.rename({"phony_dim_0": "p"})

    return ds, files
    
# mins_per_timeIndex = int(ModelData.dt/60)
# ten_mins = int(10/mins_per_timeIndex)
# ModelData.timeStrings = ModelData.timeStrings[::ten_mins] #doesn't work since parcel tracking crosses times

def Get_VARS_Data(ModelData, pIndices, varNames=None): 
    """
    Version for slicing a range of specific parcels
    """

    #See FUNCTIONS_Variable_Calculation/CallLagrangianArray for similar code
    dataType = "LagrangianArrays"
    dataName = "VARS"    
    dataFolder = dataName
    
    inputDataDirectory = os.path.normpath(os.path.join(DataManager.inputDirectory, "..", dataType,
                                                       f"Simulation_{ModelData.simulationNumber}_
                                                       {DataManager.res}_{DataManager.t_res}_
                                                       {DataManager.Nz_str}nz", dataFolder))

    files = [os.path.join(
                    inputDataDirectory,
                    f"{dataName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz_{timeString}.h5") for timeString in ModelData.timeStrings]
    def limitParcels(ds): 
        ds = ds.isel(phony_dim_0=pIndices)
        if varNames is not None:
                ds = ds[varNames]
        return ds
    VARS_Data,_ = OpenMultipleSingleTimes_LagrangianArray_JobArray_SpecifyFiles(files=files,
                                                                              ModelData=ModelData, start_job=None,end_job=None, 
                                                                              limitParcels=limitParcels)
    return VARS_Data

def GetVARSVariablesData(VARS_Data):
    ds = VARS_Data.compute()

    # Now extract NumPy arrays
    QV = ds['QV'].data
    THETA_v = ds['THETA_v'].data
    THETA_e = ds['THETA_e'].data
    return QV, THETA_v, THETA_e

# def MakeTestArray():
#     trackedArray = np.zeros((5,3),dtype='int')
#     trackedArray[:,0]=[1,3,5,7,9] #p
#     trackedArray[:,1]=[80,90,100,110,120] #t1
#     trackedArray[:,2]=[85,96,107,118,129] #t2
#     return trackedArray

In [ ]:
##############################################
#COMPUTING FUNCTIONS

In [ ]:
def LimitTrackedArraysRows(trackedArrays,
                           limit=10000):
    for parcelType in trackedArrays:
        for parcelDepth in trackedArrays[parcelType]:
            array = trackedArrays[parcelType][parcelDepth]
            trackedArrays[parcelType][parcelDepth]\
            = trackedArrays[parcelType][parcelDepth][:limit, :]

In [ ]:
def GetData(trackedArray):
    pIndices=trackedArray[:,0] #for example all pIndices for CL parcels
    varNames_spatial = ['W','QCQI','z']
    Lagrangian_Binary_Array_Data,_ = Get_Lagrangian_Binary_Array_Data(ModelData, pIndices, varNames=varNames_spatial)
    [W, QCQI, parcel_z] = GetSpatialData(Lagrangian_Binary_Array_Data)

    varNames_VARS = ["QV", "THETA_v", "THETA_e" ]
    VARS_Data = Get_VARS_Data(ModelData, pIndices, varNames=varNames_VARS)
    variableOutput = GetVARSVariablesData(VARS_Data)

    dataDictionary = {"W": W,"QCQI": QCQI,"parcel_z": parcel_z}
    dataDictionary.update(zip(varNames_VARS, variableOutput))
    return dataDictionary

def ComputeTrajectories(variableData,t1,t2,after):
    #making output array
    trajectoriesArray = np.full_like(variableData,fill_value=np.nan,dtype=float)

    #masking the final output array
    time_idx = np.arange(data.shape[0])[:, np.newaxis]
    mask = (time_idx >= t1) & (time_idx <= t2+after)
    
    #applying mask to output array
    trajectoriesArray[mask] = variableData[mask]

    return trajectoriesArray

In [ ]:
def RunCode(trackedArrays):

    parcelTypes=['CL','nonCL']
    parcelDepths=['SHALLOW','DEEP']
    
    #subset out parcelType and parcelDepth
    trajectoriesArrayDictionary = {pt: {pd: {} for pd in parcelDepths} for pt in parcelTypes}
    for parcelType in tqdm(parcelTypes):
        for parcelDepth in tqdm(parcelDepths, desc=f"Type: {parcelType}", leave=False):

            #subsetting data
            trackedArray = trackedArrays[parcelType][parcelDepth]
            t1 = trackedArray[:, 1].astype(int);t2 = trackedArray[:, 2].astype(int)
            after = trackedArray[:, 3].astype(int)
        
            # Data Calculations
            # loading variable data dictionary
            tqdm.write(f"Getting Data")
            dataDictionary = GetData(trackedArray)
            
            #getting data
            varNames = ['parcel_z','W','QCQI','QV','THETA_v','THETA_e'] #...
            for varName in tqdm(varNames, desc="Subsetting Variable", leave=False):
                variableData = dataDictionary[varName]
                
                #computing trajectories for each variable
                trajectoriesArray = ComputeTrajectories(variableData,t1,t2,after)
                trajectoriesArrayDictionary[parcelType][parcelDepth][varName] = trajectoriesArray

    return trajectoriesArrayDictionary

In [ ]:
##############################################
#COMPUTING
running=True #KEEP TRUE WHEN JOBARRAY IS RUNNING
# running=False 

In [ ]:
if running:
    #Loading in Tracked Parcels Info
    # trackedArray = MakeTestArray() #*testing
    trackedArrays,_ = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,Results_InputOutput_Class)
    LimitTrackedArraysRows(trackedArrays) #limiting number of parcels to allow computation to complete

    trajectoriesArrayDictionary = RunCode(trackedArrays)
    TrackedProfiles_DataLoading_CLASS.SaveProfile(ModelData,DataManager_TrackedProfiles, 
                                                  trajectoriesArrayDictionary, dataName, t='all_times')

In [ ]:
##############################################
#PLOTTING FUNCTIONS

In [ ]:
#Plotting All Trajectories

# def PlotTrajectories(trajectoriesArray,zArray): #simple version, not colored by z
#     time=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
#     plt.plot(time, trajectoriesArray)
#     plt.xlim(time[0],time[-1])
#     plt.ylim(bottom=0) #for z
# PlotTrajectories(trajectoriesArray)

from matplotlib.collections import LineCollection
def PlotTrajectories(trajectoriesArray, zArray, dataRange, cmap='turbo', yLabel=None,
                     vmaxMethod='nanmax',roundingNumber=1000,
                     fig=None,axis=None):
    time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
    if fig is None:
        fig, axis = plt.subplots() # Better to create fig/ax explicitly

    # 1. Setup color scaling
    if vmaxMethod == 'nanmax':
        vmaxValue = np.nanmax(zArray)
    elif vmaxMethod == 'percentile99':
        vmaxValue = np.nanpercentile(zArray,99)
    elif isinstance(vmaxMethod, (int, float)):
        vmaxValue = vmaxMethod
    if roundingNumber is not None:
        vmaxValue = np.ceil(vmaxValue/roundingNumber)*roundingNumber

    norm = plt.Normalize(vmin=0, vmax=vmaxValue)

    for i in tqdm(range(trajectoriesArray.shape[1])):
        y = trajectoriesArray[:, i]
        z = zArray[:, i]
        
        mask = ~np.isnan(y)
        if not np.any(mask): continue
        
        t_m, y_m, z_m = time[mask], y[mask], z[mask]

        # 2. Reshape into segments
        points = np.array([t_m, y_m]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)

        # 3. Create and add collection
        lc = LineCollection(segments, cmap=cmap, norm=norm, alpha=0.6)
        lc.set_array(z_m) 
        axis.add_collection(lc)

    # 4. Final touches
    axis.set_xlim(time[0], time[-1])
    axis.set_xlabel('time (hrs)')
    if yLabel is not None:
        axis.set_ylabel(yLabel)

    #setting ylim
    if dataRange is None:
        dataMin=np.nanmin(trajectoriesArray)
        dataMax=np.nanmax(trajectoriesArray)
        dataRange=(dataMin,dataMax)
    axis.set_ylim(dataRange)
    
    # FIXED: Tell the colorbar which axes to use (ax=axis)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    fig.colorbar(sm, ax=axis, label='z (m)', extend='max')

dataRanges = {
    "SHALLOW": {
        "parcel_z": (0, 5000),
        "QV": (5/1e3, 20/1e3),
        "THETA_v": (305, 320),
        "THETA_e": (330, 348),
        "W": (0, 12),
        "QCQI": (0, 3.5e-3)
    },
    "DEEP": {
        "parcel_z": (0, 13000),
        "QV": (5/1e3, 20/1e3),
        "THETA_v": (305, 345),
        "THETA_e": (330, 348),
        "W": (0, 18),
        "QCQI": (0, 5/1e3)
    }
}

In [ ]:
# # Plotting All Trajectories

# from matplotlib.colors import LogNorm, Normalize
# def PlotDensity(trajectoriesArray,zArray,varName nBins=40, yLabel=None,
#                 fig=None, axis=None, 
#                 logNorm=False, lowerThreshold=None):
#     """
#     #Plot Density By Count
#     """
#     if fig is None: fig, axis = plt.subplots()

#     # 1. Flatten data
#     time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
#     t_mesh = np.broadcast_to(time[:, None], trajectoriesArray.shape)
    
#     mask = ~np.isnan(trajectoriesArray)
#     if lowerThreshold is not None:
#         mask = mask & (trajectoriesArray > lowerThreshold)
#     t_flat, z_flat = t_mesh[mask], trajectoriesArray[mask]

#     # 2. Fix the Norm
#     # We want to scale the COLOR based on the number of particles (counts)
#     if logNorm:
#         norm = LogNorm() # Automatically scales based on count density
#     else:
#         norm = Normalize()

#     # 3. Plot
#     counts, xedges, yedges, im = axis.hist2d(
#         t_flat, z_flat, bins=[ModelData.Ntime // 2, nBins], 
#         cmap='turbo', norm=norm, cmin=1
#     )
    
#     fig.colorbar(im, ax=axis, label='Particle Count')
    
#     axis.set_xlabel('Time (h)')
#     if yLabel is not None: axis.set_ylabel(yLabel)
#     if not np.all(np.isnan(trajectoriesArray)):
#         axis.set_ylim(bottom=np.nanmin(trajectoriesArray) )
#         if "THETA" in varName:
#             ylimTop=np.nanpercentile(trajectoriesArray,95)*1.1
#         else:
#             ylimTop=np.nanmax(trajectoriesArray)*1.1
#         axis.set_ylim(top=ylimTop)

# def PlotDensity(trajectoriesArray, zArray,dataRange, nBins=40, yLabel=None,
#                 fig=None, axis=None, logNorm=False, lowerThreshold=None,
#                 vmaxMethod='nanmax',roundingNumber=1000):
#     """
#     #Plot Density By Height
#     #2D Histogram Version
#     """
#     if fig is None: fig, axis = plt.subplots()

#     if vmaxMethod == 'nanmax':
#         vmaxValue = np.nanmax(zArray)
#     elif vmaxMethod == 'percentile99':
#         vmaxValue = np.nanpercentile(zArray,99)
#     elif isinstance(vmaxMethod, (int, float)):
#         vmaxValue = vmaxMethod
#     if roundingNumber is not None:
#         vmaxValue = np.ceil(vmaxValue/roundingNumber)*roundingNumber

#     # 1. Flatten data
#     time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
#     t_mesh = np.broadcast_to(time[:, None], trajectoriesArray.shape)
    
#     mask = ~np.isnan(trajectoriesArray) & ~np.isnan(zArray)
#     if lowerThreshold is not None:
#         mask = mask & (trajectoriesArray > lowerThreshold)
        
#     t_flat = t_mesh[mask]
#     y_flat = trajectoriesArray[mask]
#     z_flat = zArray[mask]

#     # 2. Manually calculate the Binned Mean
#     # Define bins
#     x_bins = np.linspace(time[0], time[-1], ModelData.Ntime // 2)
#     y_bins = np.linspace(np.nanmin(y_flat), np.nanmax(y_flat), nBins)

#     # Sum of Z values in each bin
#     z_sum, x_edges, y_edges = np.histogram2d(t_flat, y_flat, bins=[x_bins, y_bins], weights=z_flat)
    
#     # Count of particles in each bin
#     counts, _, _ = np.histogram2d(t_flat, y_flat, bins=[x_bins, y_bins])

#     # Calculate Average (handle division by zero where counts=0)
#     with np.errstate(divide='ignore', invalid='ignore'):
#         z_mean = z_sum / counts

#     # 3. Plot using pcolormesh (which creates the 'hist2d' look)
#     # Note: We transpose z_mean because pcolormesh expects (Y, X) or similar alignment
#     im = axis.pcolormesh(x_edges, y_edges, z_mean.T, cmap='turbo', shading='auto',
#                         vmin=0, vmax=vmaxValue)
    
#     # 4. Final touches
#     fig.colorbar(im, ax=axis, label='z (m)')
    
#     axis.set_xlabel('Time (h)')
#     #setting ylim
#     if dataRange is None:
#         dataMin=np.nanmin(trajectoriesArray)
#         dataMax=np.nanmax(trajectoriesArray)
#         dataRange=(dataMin,dataMax)
#     axis.set_ylim(dataRange)

def PlotDensity(trajectoriesArray, zArray, dataRange, nBins=40, yLabel=None,
                fig=None, axis=None, logNorm=False, lowerThreshold=None,
                 vmaxMethod='nanmax',roundingNumber=1000):
    """
    #Plot Density By Height
    #HexBin Version
    """
    if fig is None: fig, axis = plt.subplots()

    if vmaxMethod == 'nanmax':
        vmaxValue = np.nanmax(zArray)
    elif vmaxMethod == 'percentile99':
        vmaxValue = np.nanpercentile(zArray,99)
    if isinstance(vmaxMethod, (int, float)):
        vmaxValue = vmaxMethod
    if roundingNumber is not None:
        vmaxValue = np.ceil(vmaxValue/roundingNumber)*roundingNumber

    # 1. Flatten data
    time = np.arange(ModelData.Ntime) * ModelData.dt / 3600 + 6
    t_mesh = np.broadcast_to(time[:, None], trajectoriesArray.shape)
    
    # Create mask for valid data and thresholds
    mask = ~np.isnan(trajectoriesArray) & ~np.isnan(zArray)
    if lowerThreshold is not None:
        mask = mask & (trajectoriesArray > lowerThreshold)
        
    t_flat = t_mesh[mask]
    y_flat = trajectoriesArray[mask]
    z_flat = zArray[mask] # The values we want to average for the color

    # 2. Use hexbin with 'C' parameter
    # C=z_flat tells hexbin to aggregate height values
    # reduce_C_function=np.nanmean calculates the average height in each bin
    hb = axis.hexbin(
        t_flat, y_flat, C=z_flat, 
        reduce_C_function=np.nanmean,
        gridsize=(ModelData.Ntime // 2, nBins),
        cmap='turbo', 
        mincnt=1, # Don't plot empty bins
        vmin=0,
        vmax=vmaxValue
    )
    
    # 3. Final touches
    fig.colorbar(hb, ax=axis, label='z (m)', extend='max')
    
    #setting ylim
    if dataRange is None:
        dataMin=np.nanmin(trajectoriesArray)
        dataMax=np.nanmax(trajectoriesArray)
        dataRange=(dataMin,dataMax)
    axis.set_ylim(dataRange)

def CombinedTrajectoryPlot(trajectoriesArrayDictionary, parcelType, parcelDepth, varNames):
    """
    Plots trajectories and density for a list of variables in a grid.
    """
    
    # 1. Configuration for specific variables
    # Default is logNorm=False unless specified here
    plot_configs = {
        'parcel_z': {'logNorm': False},
        'W':        {'logNorm': True, 'lowerThreshold': 0.1},
        'QCQI':     {'logNorm': True, 'lowerThreshold': 1e-6}
    }

    # 2. Setup Figure
    fig, axes = plt.subplots(len(varNames), 2, figsize=(10, 4 * len(varNames)), gridspec_kw={"wspace": 0.4},squeeze=False)
    trajectoriesArray = trajectoriesArrayDictionary[parcelType][parcelDepth]
    zArray = trajectoriesArray['parcel_z']

    # 3. Loop through variables and plot
    for i, varName in enumerate(varNames):
        config = plot_configs.get(varName, {'logNorm': False})
        dataRange = dataRanges[parcelDepth][varName]
        vmaxMethod = dataRanges[parcelDepth]['parcel_z'][1]

        # Column 0: Trajectories
        PlotTrajectories(trajectoriesArray[varName], zArray,dataRange, yLabel=varName,
                         fig=fig, axis=axes[i, 0],vmaxMethod=vmaxMethod)

        # Column 1: Density
        PlotDensity(trajectoriesArray[varName], zArray,dataRange, yLabel=varName,
                    fig=fig,axis=axes[i, 1],vmaxMethod=vmaxMethod,**config)

    return fig

In [ ]:
#Plotting All Trajectories Mean

def PlotTrajectories_Mean(trajectoriesArray_mean,dataRange,
                          yLabel=None,color='green',linestyle='-',
                          fig=None,axis=None): #mean version, may be useful later

    if fig is None:
        fig,axis=plt.subplots()
    time=np.arange(ModelData.Ntime)*ModelData.dt/3600+6
    axis.plot(time, trajectoriesArray_mean,color=color,linestyle=linestyle)
    axis.set_xlim(time[0],time[-1])
    axis.set_ylim(bottom=0) #for z

    axis.set_xlabel('Time (hrs)')
    if yLabel is not None: axis.set_ylabel(yLabel)

    #setting ylim
    if dataRange is None:
        dataMin=np.nanmin(trajectoriesArray)
        dataMax=np.nanmax(trajectoriesArray)
        dataRange=(dataMin,dataMax)
    axis.set_ylim(dataRange)

def CombinedTrajectoryMeanPlot(trajectoriesArrayDictionary, parcelTypes, parcelDepth, varNames):
    """
    Plots the mean trajectories for multiple parcel types across specified variables.
    """
    # 1. Setup Figure
    fig, axes = plt.subplots(len(varNames), 1, figsize=(10, 4 * len(varNames)), 
                             gridspec_kw={"wspace": 0.4},squeeze=False)
    
    # 2. Determine color based on depth
    color = 'green' if parcelDepth == "SHALLOW" else 'blue'
    
    # 3. Loop through each variable (one row per variable)
    for i, varName in enumerate(varNames):
        axis = axes[i, 0]
        
        # 4. Overlay each parcel type on the current axis
        for parcelType in parcelTypes:
            dataRange = dataRanges[parcelDepth][varName]
            # Determine linestyle
            linestyle = '-' if parcelType == "CL" else '--'
            
            # Extract data
            trajectoriesArray = trajectoriesArrayDictionary[parcelType][parcelDepth]
            
            # Calculate mean and plot, ignoring nan warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                
                mean_data = np.nanmean(trajectoriesArray[varName], axis=1)
                
                PlotTrajectories_Mean(mean_data,dataRange,yLabel=varName, 
                                      color=color,linestyle=linestyle,
                                      fig=fig,axis=axis)
    return fig

In [ ]:
def CalculateAverageParcelLifeCyle(data):
    # data.shape is (133, 4230)
    n_times, n_parcels = data.shape
    
    # Create an empty container of the same shape filled with NaNs
    aligned_data = np.full((n_times, n_parcels), np.nan)
    
    for i in range(n_parcels):
        parcel_slice = data[:, i]
        
        # Find where the data is NOT nan
        mask = ~np.isnan(parcel_slice)
        
        if np.any(mask):
            # Extract only the "burst" of real data
            valid_data = parcel_slice[mask]
            
            # Place this burst at the beginning of our aligned array
            # This makes "Model Time X" become "Time Since Birth 0"
            aligned_data[:len(valid_data), i] = valid_data
            
    # Calculate the mean across parcels for each "relative" time step
    # As time increases, np.nanmean ignores the NaNs from shorter parcels
    mean_profile = np.nanmean(aligned_data, axis=1)
    # mean_profile = np.nanmax(aligned_data, axis=1)
    # mean_profile = np.nanpercentile(aligned_data,99, axis=1)
    
    # Calculate the count of active parcels at each relative time step
    counts = np.sum(~np.isnan(aligned_data), axis=1)

    return aligned_data,mean_profile, counts


def PlotAverageParcelLifeCyle_Mean(aligned,mean,counts,varName,
                                   label,color,linestyle,
                                   fig=None,axis=None, yLabel=None):
    
    # 2. Calculate Standard Error (Uncertainty of the mean)
    std = np.nanstd(aligned, axis=1); sem = std / np.sqrt(counts)
    mask = counts >= 1
    
    # 3. Plotting
    if fig is None:
        fig, axis = plt.subplots(figsize=(8, 5))
    
    t = np.arange(len(mean))*ModelData.dt/60    
    
    # Plot CL
    axis.plot(t[mask], mean[mask], color=color,linestyle=linestyle,label=label)
    axis.fill_between(t[mask], mean[mask] - sem[mask], 
                    mean[mask] + sem[mask], alpha=0.2)
   
    axis.set_xlabel("Minutes Since Birth"); 
    if yLabel is not None: axis.set_ylabel(yLabel)
    axis.legend()
    axis.set_title("Average Parcel Life Cycle")
    axis.set_xlim(left=0);#axis.set_ylim(bottom=0)
    
    if varName=="W":
        axhlineValue = 0.1
        axis.axhline(axhlineValue,color='gray',linestyle='--',zorder=-50)
    elif varName=="QCQI":
        axhlineValue = 1e-6
        axis.axhline(axhlineValue,color='gray',linestyle='--',zorder=-50)
def CombinedAverageParcelLifeCyleMeanPlot(trajectoriesArrayDictionary, 
                                         parcelTypes, parcelDepth, varNames):
    """
    Plots the average lifecycle mean for multiple parcel types across specified variables.
    """
    # 1. Setup Figure
    # squeeze=False ensures axes[i, 0] always works
    fig, axes = plt.subplots(len(varNames), 1, figsize=(10, 4 * len(varNames)), 
                             gridspec_kw={"hspace": 0.4},squeeze=False)
    
    # 2. Determine color based on depth
    color = 'green' if parcelDepth == "SHALLOW" else 'blue'
    
    # 3. Outer Loop: Variables (Rows)
    for i, varName in enumerate(varNames):
        axis = axes[i, 0]
        
        # 4. Inner Loop: Parcel Types (Lines on the same plot)
        for parcelType in parcelTypes:
            linestyle = '-' if parcelType == "CL" else '--'
            label = f'{parcelType} {parcelDepth}'
            
            # Extract data specifically for this type/depth/variable
            data = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]

            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                
                # Calculate the aligned lifecycle metrics
                aligned, mean, counts = CalculateAverageParcelLifeCyle(data)
                
                # Plot onto the current axis
                PlotAverageParcelLifeCyle_Mean(aligned, mean, counts, varName,
                                               fig=fig, axis=axis, 
                                               yLabel=varName,label=label, color=color, linestyle=linestyle)
    return fig

In [ ]:
def GetPlottingDirectory(plotFileName, plotType, folderName):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"Simulation_{ModelData.simulationNumber}_
                                             {ModelData.res}_{ModelData.t_res}_
                                             {ModelData.Nz_str}nz",folderName)
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

def SaveFigure(fig,plotType, folderName,fileName):
    plotFileName = f"{fileName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType, folderName)
    print(f"Saving figure to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')
    plt.close(fig)
def SaveAscentTrajectoryPlot(fig,parcelTypes,parcelType,parcelDepth,dataName):
    folderName=f"{parcelTypes[0]}_vs_{parcelTypes[1]}"
    fileName=f"Tracked_Ascent_Trajectories_{dataName}_{parcelType}_{parcelDepth}" 
    SaveFigure(fig,plotType=f"Project_Algorithms/Tracked_Profiles/Tracked_Ascent_Trajectories_{dataName}",folderName=folderName,fileName=fileName)

In [ ]:
##############################
#PLOTTING

In [ ]:
trajectoriesArrayDictionary = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t='all_times')
varNames = ['parcel_z','QV','THETA_v','THETA_e','W','QCQI']

In [ ]:
parcelDepth="SHALLOW"

parcelType="CL"
fig=CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType=parcelType,parcelDepth=parcelDepth,varNames=varNames)
SaveAscentTrajectoryPlot(fig,parcelTypes,parcelType,parcelDepth,dataName)

parcelType="nonCL"
fig=CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType=parcelType,parcelDepth=parcelDepth,varNames=varNames)
SaveAscentTrajectoryPlot(fig,parcelTypes,parcelType,parcelDepth,dataName)

In [ ]:
parcelDepth="DEEP"

parcelType="CL"
fig=CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType=parcelType,parcelDepth=parcelDepth,varNames=varNames)
SaveAscentTrajectoryPlot(fig,parcelTypes,parcelType,parcelDepth,dataName)

parcelType="nonCL"
fig=CombinedTrajectoryPlot(trajectoriesArrayDictionary,
                           parcelType=parcelType,parcelDepth=parcelDepth,varNames=varNames)
SaveAscentTrajectoryPlot(fig,parcelTypes,parcelType,parcelDepth,dataName)

In [ ]:
##############################
#PLOTTING NOT SO INFORMATIVE PLOTS

In [ ]:
#These plots aren't very explanatory, see next
fig=CombinedTrajectoryMeanPlot(trajectoriesArrayDictionary,
                               parcelTypes=["CL","nonCL"],
                               parcelDepth="SHALLOW",varNames=varNames)

fig=CombinedTrajectoryMeanPlot(trajectoriesArrayDictionary,
                               parcelTypes=["CL","nonCL"],
                               parcelDepth="DEEP",varNames=varNames)


In [ ]:
#These average over all times leading to incorrect conclusions, see the next test which splits by time
fig = CombinedAverageParcelLifeCyleMeanPlot(trajectoriesArrayDictionary,
                                            parcelTypes=['CL','nonCL'],
                                            parcelDepth='SHALLOW',varNames=varNames)

fig = CombinedAverageParcelLifeCyleMeanPlot(trajectoriesArrayDictionary,
                                            parcelTypes=['CL','nonCL'],
                                            parcelDepth='DEEP',varNames=varNames)

In [ ]:
def GetParcelsByBirthWindow(data, start_min, end_min):
    """
    Returns only the parcels that were born between start_min and end_min.
    """
    # Convert minutes to model indices
    start_idx = int(start_min / (ModelData.dt / 60))
    end_idx = int(end_min / (ModelData.dt / 60))
    
    # Find the birth index (first non-nan) for every parcel
    mask = ~np.isnan(data)
    # argmax returns the first index of True; we check if that's in our window
    birth_indices = np.where(mask.any(axis=0), mask.argmax(axis=0), -1)
    
    in_window = (birth_indices >= start_idx) & (birth_indices < end_idx)
    return data[:, in_window]


def CombinedEvolutionaryLifeCyclePlot(trajectoriesArrayDictionary, parcelTypes, 
                                     parcelDepth, varNames, time_windows):
    """
    time_windows: List of tuples [(0, 60), (60, 120), (120, 180)] in minutes.
    """
    fig, axes = plt.subplots(len(varNames), 1, figsize=(10, 5 * len(varNames)), 
                             gridspec_kw={"hspace": 0.4}, squeeze=False)
    
    # Base color (we will adjust brightness/alpha per window)
    base_color = 'green' if parcelDepth == "SHALLOW" else 'blue'
    
    for i, varName in enumerate(varNames):
        axis = axes[i, 0]
        
        for parcelType in parcelTypes:
            linestyle = '-' if parcelType == "CL" else '--'
            
            for j, (t_start, t_end) in enumerate(time_windows):
                # 1. Get raw data
                full_data = trajectoriesArrayDictionary[parcelType][parcelDepth][varName]
                
                # 2. Filter for parcels born in this specific window
                window_data = GetParcelsByBirthWindow(full_data, t_start, t_end)
                
                if window_data.shape[1] == 0:
                    continue # Skip if no parcels were born in this window
                
                # 3. Calculate aligned lifecycle
                aligned, mean, counts = CalculateAverageParcelLifeCyle(window_data)
                
                # 4. Plot with evolving alpha or color intensity
                alpha = 0.3 + (j / len(time_windows)) * 0.7 # Gets darker/more solid over time
                label = f"{parcelType} ({t_start}-{t_end}m)"
                
                # Reuse your PlotAverageParcelLifeCyle_Mean logic
                # We pass alpha inside to show the evolution
                PlotAverageParcelLifeCyle_Mean(
                    aligned, mean, counts, varName,
                    fig=fig, axis=axis, yLabel=varName,
                    label=label, color=base_color, linestyle=linestyle
                )
                # Adjust alpha of the last added line
                axis.get_lines()[-1].set_alpha(alpha)
                
    return fig

# # Define 1-hour windows for the first 3 hours
# hourSize=1
# windows = [(t, t + hourSize*60) for t in range(0, 11 * 60, hourSize*60)]

# varNames = ['parcel_z', 'QV','THETA_v']

# fig = CombinedEvolutionaryLifeCyclePlot(
#     trajectoriesArrayDictionary,
#     parcelTypes=['CL', 'nonCL'],
#     parcelDepth='SHALLOW',
#     varNames=varNames,
#     time_windows=windows
# )


import matplotlib.pyplot as plt
import numpy as np
import warnings

def CombinedEvolutionaryDifferencePlot(trajectoriesArrayDictionary, parcelTypes, 
                                       parcelDepth, varNames, time_windows):
    """
    Plots the difference (parcelTypes[0] - parcelTypes[1]) across different birth windows.
    Labels intervals in hours for better readability.
    """
    fig, axes = plt.subplots(len(varNames), 1, figsize=(10, 5 * len(varNames)), 
                             gridspec_kw={"hspace": 0.4}, squeeze=False)
    
    # Standard Matplotlib color cycle
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
    
    for i, varName in enumerate(varNames):
        axis = axes[i, 0]
        
        for j, (t_start, t_end) in enumerate(time_windows):
            # 1. Access both datasets
            # Logic: parcelTypes[0] is usually 'CL', parcelTypes[1] is 'nonCL'
            data0 = trajectoriesArrayDictionary[parcelTypes[0]][parcelDepth][varName]
            data1 = trajectoriesArrayDictionary[parcelTypes[1]][parcelDepth][varName]
            
            # 2. Filter parcels born in this window
            window_data0 = GetParcelsByBirthWindow(data0, t_start, t_end)
            window_data1 = GetParcelsByBirthWindow(data1, t_start, t_end)
            
            # Skip if one group is empty in this window
            if window_data0.shape[1] == 0 or window_data1.shape[1] == 0:
                continue
                
            # 3. Calculate means (suppressing warnings for empty tail-end slices)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=RuntimeWarning)
                _, mean0, _ = CalculateAverageParcelLifeCyle(window_data0)
                _, mean1, _ = CalculateAverageParcelLifeCyle(window_data1)
            
            # 4. Calculate Difference
            # Ensure arrays match length in case of different max lifespans
            min_len = min(len(mean0), len(mean1))
            diff = mean0[:min_len] - mean1[:min_len]
            
            # 5. Plotting
            t_relative = np.arange(min_len) * ModelData.dt / 60
            line_color = colors[j % len(colors)]
            
            # Convert minutes to hours for the label
            h_start = t_start / 60 + 6
            h_end = t_end / 60 + 6
            label = fr"$\Delta$ {h_start:g}-{h_end:g}h"
            
            axis.plot(t_relative, diff, color=line_color, linewidth=2, label=label)

        # Formatting each subplot
        axis.axhline(0, color='black', lw=1.5, ls=':', alpha=0.6, zorder=0)
        axis.set_ylabel(f"Diff ({varName})")
        axis.set_xlabel("Minutes Since Birth")
        axis.set_title(f"Evolution of Difference: {parcelTypes[0]} - {parcelTypes[1]} ({parcelDepth})")
        axis.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small', title="Birth Window")
                
    return fig

# --- Execution ---
hourSize = 1
windows = [(t, t + hourSize*60) for t in range(0, 11 * 60, hourSize*60)]
varNames = ['parcel_z', 'QV', 'THETA_v']

fig = CombinedEvolutionaryDifferencePlot(
    trajectoriesArrayDictionary,
    parcelTypes=['CL', 'nonCL'],
    parcelDepth='SHALLOW',
    varNames=varNames,
    time_windows=windows
)
plt.show()

In [ ]:
############################
#TESTING

In [ ]:
# #testing min parcel ascent initial W
# a=trajectoriesArrayDictionary['CL']['SHALLOW']['W']

# lst=[]
# for i in range(a.shape[1]):
#     b=a[:,i]
#     c=b[~np.isnan(b)][1]
#     lst.append(c)

# d=np.array(lst)
# d[d<0]